# 11 | Matched Random Forest vs Temporal CNN

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Compare classical Random Forest and raw-signal 1D-CNN with identical source IMU windows, participant groups, eight-class target and Validation rows.

**Important:** These are different input representations: Random Forest receives 30 deterministic features extracted from each exact CNN window; CNN receives the original six-channel sequence. This is a matched data/protocol comparison, **not** a controlled architecture-only causal comparison or final held-out benchmark. Original Test participants S023 and S032 are not loaded.

## 01. Imports and reusable code

All source processing, classical model fitting and PyTorch training remain in `.py` modules. Each code cell contains brief purpose comments.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, ConfusionMatrixDisplay, confusion_matrix

# 1. Resolve the project root and register the existing source modules.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Open this notebook within human-movement-intelligence.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

# 2. Reuse the existing dataset, window, features, split and model code.
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS, prepare_harth_window_cohort
from core.data_splitting import split_group_indices
from core.feature_engineering import extract_signal_features
from core.temporal_cnn import train_temporal_cnn
from core.ml_baselines import evaluate_predictions_by_subject, classification_class_report
from core.group_cross_validation import compare_models_by_subject

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

## 02. Reuse exactly the Stage 10 raw-window cohort

Keep the same 29 development subjects, the same limit of 500 windows per person, and the same eight-class scope. Out-of-task labels are excluded only after time-safe windowing. The two original Test participants are never loaded.

In [ ]:
# 1. Fix identical cohort and sampling parameters before model fitting.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
WINDOW_CAP = 500
assert len(DEVELOPMENT_SUBJECTS) == 29

# 2. Generate the original six-channel signal windows once for both models.
X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    max_windows_per_subject=WINDOW_CAP,
    window_size=100, step_size=50, sampling_rate=50,
)
assert X.shape[0] == len(y) == len(metadata)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
print("Shared raw windows:", X.shape)
display(overview[["subject_id", "eligible_windows", "sampled_windows", "observed_sampled_classes"]])

## 03. Freeze identical Train and Validation participants

Use exactly the deterministic Stage 10 group split. A third partition consists of *unused development subjects* and is excluded from both model fits and comparisons.

In [ ]:
# 1. Recreate the same participant-disjoint split as Stage 10.
parts = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.70, validation_fraction=0.20,
    test_fraction=0.10, random_state=42,
)
train_idx = parts["train"]
val_idx = parts["validation"]
group_members = {
    name: set(metadata.iloc[rows]["subject_id"])
    for name, rows in parts.items()
}
assert not (group_members["train"] & group_members["validation"])
assert not (group_members["train"] & group_members["test"])
assert not (group_members["validation"] & group_members["test"])

# 2. Preserve one source-row map for both model input representations.
X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]
val_subject_ids = metadata.iloc[val_idx]["subject_id"].to_numpy()
assert X_val.shape[0] == len(y_val) == len(val_subject_ids)
display(pd.DataFrame([
    {"partition": name, "subjects": len(group_members[name]), "windows": len(rows),
     "IDs": ", ".join(sorted(group_members[name]))}
    for name, rows in parts.items()
]))

## 04. Generate Random Forest inputs from the exact same windows

Random Forest uses five deterministic features for each of six channels. No new windows or independently selected examples are introduced.

In [ ]:
# 1. Extract the established features without using labels or participant IDs.
feature_table = extract_signal_features(
    X, channel_names=SENSOR_COLUMNS,
    methods=["mean", "std", "rms", "peak_to_peak", "waveform_length"],
)
X_rf_train = feature_table.iloc[train_idx]
X_rf_val = feature_table.iloc[val_idx]

# 2. Verify exact row alignment and report the two model input shapes.
assert len(X_rf_train) == len(X_train)
assert len(X_rf_val) == len(X_val)
assert feature_table.shape == (len(X), 30)
print("CNN Train:", X_train.shape, "| Validation:", X_val.shape)
print("RF Train:", X_rf_train.shape, "| Validation:", X_rf_val.shape)

## 05. Fit the matched Random Forest baseline

Reuse the same Random Forest hyperparameters from Stage 9, fitting exclusively on Stage 10's Train subjects. No Validation labels influence model fitting.

In [ ]:
# 1. Build a fresh RF estimator with unchanged research parameters.
rf = RandomForestClassifier(
    n_estimators=120, min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1,
)
rf.fit(X_rf_train, y_train)
rf_prediction = rf.predict(X_rf_val)
assert len(rf_prediction) == len(y_val)
rf_f1 = f1_score(y_val, rf_prediction, labels=TASK_CLASSES,
                 average="macro", zero_division=0)
print("RF matched Validation Macro F1:", round(rf_f1, 4))

## 06. Fit the temporal CNN on the identical participant split

The CNN receives the original windows. Each channel's mean/SD is estimated from Train only. Epochs, batch size and learning rate match the Stage 10 pilot. The last epoch is reported rather than selecting the highest Validation score.

In [ ]:
# 1. Train the raw-signal CNN only using the same Train and Validation rows.
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cnn, history, cnn_prediction, scaler = train_temporal_cnn(
    X_train=X_train, y_train=y_train,
    X_validation=X_val, y_validation=y_val,
    classes=TASK_CLASSES,
    epochs=EPOCHS, batch_size=128, learning_rate=1e-3,
    seed=42, device=DEVICE,
)
assert len(cnn_prediction) == len(rf_prediction) == len(y_val)
display(history.round(4))

## 07. Compare validation metrics and class-wise performance

Both models are evaluated on exactly the same held-out windows and labels. Macro F1 is the primary aggregate measure; Balanced Accuracy and per-class F1 reveal class imbalance effects.

In [ ]:
# 1. Compute identical metrics on the same Validation labels.
predictions = {
    "Random Forest": rf_prediction,
    "Temporal CNN": cnn_prediction,
}
comparison = pd.DataFrame([
    {"model": name,
     "macro_f1": f1_score(y_val, prediction, labels=TASK_CLASSES,
                         average="macro", zero_division=0),
     "balanced_accuracy": balanced_accuracy_score(y_val, prediction),
     "validation_windows": len(y_val)}
    for name, prediction in predictions.items()
])
display(comparison.round(4))

# 2. Display per-class F1 side by side, including rare activities.
reports = []
for name, prediction in predictions.items():
    item = classification_class_report(
        y_val, prediction, classes=TASK_CLASSES,
        class_names=[ACTIVITY_NAMES[k] for k in TASK_CLASSES],
    )
    item["model"] = name
    reports.append(item)
class_table = pd.concat(reports, ignore_index=True)
display(class_table[["model", "class", "support", "precision", "recall", "f1"]].round(3))

# 3. Use one grouped chart for the same per-class score scale.
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(8)
for i, name in enumerate(predictions):
    values = class_table.loc[class_table["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i - 0.5) * 0.36, values, width=0.34, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
              rotation=35, ha="right")
ax.set(ylabel="Validation F1", title="Matched class-wise F1: RF versus CNN",
       ylim=(0, 1))
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## 08. Inspect the two confusion matrices in one Figure

Confusion matrices reveal whether either model misses Shuffling or stair activities. They are validation diagnostics, not final Test results.

In [ ]:
# Plot confusion matrices with identical true/predicted axes.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, prediction) in zip(axes, predictions.items()):
    matrix = confusion_matrix(y_val, prediction, labels=TASK_CLASSES)
    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=[ACTIVITY_NAMES[c] for c in TASK_CLASSES],
    ).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"{name}: matched Validation")
fig.tight_layout()
plt.show()

## 09. Compare both models for each held-out participant

This analysis treats each independent participant once, avoiding claims that overlapping windows are independent test subjects. Paired differences are descriptive only; no p-value is claimed.

In [ ]:
# 1. Calculate fixed eight-class Macro F1 separately for each Validation subject.
subject_results = pd.concat([
    evaluate_predictions_by_subject(
        y_true=y_val, y_pred=prediction, subject_ids=val_subject_ids,
        classes=TASK_CLASSES, model_name=name,
    )
    for name, prediction in predictions.items()
], ignore_index=True)
display(subject_results.sort_values(["subject_id", "model"]).round(4))

# 2. Compare the two model scores for exactly the same held-out people.
paired, paired_summary = compare_models_by_subject(
    subject_results, model_a="Temporal CNN",
    model_b="Random Forest", score_column="macro_f1_fixed_task",
)
display(paired.round(4))
display(pd.DataFrame([paired_summary]).round(4))

# 3. Plot the signed CNN minus RF difference, one bar per participant.
fig, ax = plt.subplots(figsize=(10, 4))
difference = paired["difference_a_minus_b"]
ax.bar(paired["subject_id"], difference,
       color=["tab:blue" if d >= 0 else "tab:orange" for d in difference])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Matched held-out subject Macro F1: CNN minus Random Forest",
       xlabel="Validation participant", ylabel="Paired Macro F1 difference")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

## 10. Research quality gate

Equalising source windows and participant membership makes the validation comparison interpretable. However, model representations differ, the subset is capped at 500 windows per person, and the results are neither nested CV nor an untouched final Test benchmark.

In [ ]:
# Confirm all predictions, per-subject reports and source rows are aligned.
assert len(y_val) == len(rf_prediction) == len(cnn_prediction)
assert not (set(val_subject_ids) & RESERVED_TEST)
assert set(subject_results["subject_id"]) == set(val_subject_ids)
assert set(paired["subject_id"]) == set(val_subject_ids)
assert len(group_members["validation"]) == len(paired)
assert comparison["macro_f1"].between(0, 1).all()
assert np.isfinite(X_rf_val.to_numpy()).all()
print("Matched source windows / split / labels: PASSED")
print("Original Test loaded or evaluated: NO")
print("Model family difference conclusively established: NO")